# 🎛 Beat Mangler
**Essentia-powered beat manipulation for audio & video.**

| Mode | What it does |
|------|--------------|
| `remove` | Drops every other beat (half-time feel) |
| `swap` | Swaps beats 2 & 4 in every bar of 4 |
| `reverse` | Plays beats back in reverse order |
| `shuffle` | Randomises all beat positions |
| `repeat` | Stutters — each beat plays N times in a row |
| `interleave` | Alternates beats from two different files |

▶ **Run all cells top-to-bottom** (Runtime → Run all), or step through each cell manually.

In [ ]:
#@title 1. Install dependencies { display-mode: "form" }

import subprocess, sys, os

subprocess.run(["apt-get", "install", "-q", "-y", "ffmpeg"], capture_output=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "essentia", "moviepy", "pydub", "numpy", "tqdm"],
    check=True,
)

REPO_URL = "https://github.com/MedicDoesStuff/beat-mangler.git"
REPO_DIR = "/content/beat-mangler"

if os.path.isdir(REPO_DIR):
    print("↻  Repo already present — pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    print("⬇  Cloning repo...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Done.")

In [ ]:
#@title 2. Upload your files { display-mode: "form" }

from google.colab import files

uploaded = files.upload()

uploaded_paths = list(uploaded.keys())
for p in uploaded_paths:
    print(f"   ✓  {p}")

In [ ]:
#@title 3. Run { display-mode: "form" }

#@markdown ### Mode
MODE = "remove" #@param ["remove", "swap", "reverse", "shuffle", "repeat", "interleave"]
#@markdown ---
#@markdown ### Output format
OUTPUT_FMT = "flac" #@param ["flac", "mp3"]
#@markdown ---
#@markdown ### Repeat mode options
#@markdown *(only used when Mode = `repeat`)*
REPEAT_TIMES = 2 #@param {type:"slider", min:2, max:8, step:1}
#@markdown ---
#@markdown ### Interleave mode options
#@markdown *(only used when Mode = `interleave` — upload two files in Step 2)*
INTERLEAVE_GROUP = 1 #@param {type:"slider", min:1, max:4, step:1}

# ─────────────────────────────────────────────────────────────────────────────

import beat_mangler as fakers
from IPython.display import Audio, display

path_a = uploaded_paths[0]
path_b = uploaded_paths[1] if len(uploaded_paths) > 1 else None

if MODE == "interleave":
    if path_b is None:
        raise ValueError("Interleave mode needs two uploaded files — re-run Step 2 and upload both.")
    out = fakers.process_interleave(path_a, path_b, INTERLEAVE_GROUP, OUTPUT_FMT)

elif fakers.is_video(path_a):
    out = fakers.process_video(path_a, MODE, REPEAT_TIMES)

else:
    out = fakers.process_audio(path_a, MODE, OUTPUT_FMT, REPEAT_TIMES)

fakers.preview(out, file_is_video=fakers.is_video(out))